# DistilBERT, re-run on the project's standard 20% split

**Why this notebook exists.** `03_distilbert.ipynb` is Ryan's original run and
scores **0.9487 accuracy / 0.8951 macro F1** - the best number in the project.
But it held out **15%** of `train.json` for validation, while
`src/run_baseline.py` and `02_svm_mlp.ipynb` both hold out **20%**. The models are
therefore scored on different reviews, and the report cannot honestly put them in
one table.

This notebook runs the *same* model and the *same* hyperparameters at
`test_size=0.2`, so DistilBERT lands on the identical 1,481-review split as
everything else. It writes `results/distilbert_metrics.json`, which
`src/compare_models.py` picks up automatically and promotes to a fully
comparable row.

**Run this on a GPU.** `Runtime` -> `Change runtime type` -> **T4 GPU**. About two
minutes per epoch on a T4, against 30-60 minutes per epoch on CPU.

Nothing here is tuned. Changing a hyperparameter would defeat the purpose: the
only difference from `03_distilbert.ipynb` is the split.

## 1. Environment

In [ ]:
# Colab already has torch. Locally, install the CUDA build rather than the
# CPU one in requirements.txt, or this takes most of an hour per epoch.
%pip install -q transformers scikit-learn pandas

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    get_linear_schedule_with_warmup,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
if DEVICE.type != 'cuda':
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

## 2. Settings

Identical to `03_distilbert.ipynb` except for `TEST_SIZE`, which is the whole
point of this notebook.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 256          # 98.8% of reviews fit whole - see notebooks/01_explore_data.ipynb
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TEST_SIZE = 0.2        # <-- the only change from 03_distilbert.ipynb (was 0.15)

## 3. Data

On Colab the next cell opens a file picker: select **both** `train.json` and
`test.json`. The repo is private, so it cannot simply be cloned here. Run locally,
the notebook finds `data/` from the repo root or from `notebooks/`.

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not (Path('train.json').exists() and Path('test.json').exists()):
    print('Select BOTH train.json and test.json in the file picker.')
    files.upload()

In [ ]:
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'IE4483-sentiment-project']
ROOT = next((p for p in candidates if (p / 'data' / 'train.json').exists()), None)

if ROOT is None:
    # Colab fallback: the two json files sitting beside the notebook.
    if Path('train.json').exists():
        DATA_DIR, RESULTS_DIR = Path('.'), Path('results')
    else:
        raise FileNotFoundError(
            'train.json not found. Re-run the upload cell and select both json files.'
        )
else:
    ROOT = ROOT.resolve()
    DATA_DIR, RESULTS_DIR = ROOT / 'data', ROOT / 'results'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
train_df = pd.read_json(DATA_DIR / 'train.json')
test_df = pd.read_json(DATA_DIR / 'test.json')
print('data:', DATA_DIR, '| results:', RESULTS_DIR)
print('train', train_df.shape, '| test', test_df.shape)

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['reviews'].fillna('').tolist(),
    train_df['sentiments'].tolist(),
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=train_df['sentiments'].tolist(),
)
print(f'{len(train_texts)} train / {len(val_texts)} validation')
print('This must read 5920 train / 1481 validation to match the other models.')
assert len(val_texts) == 1481, 'split does not match src/compare_models.py'

## 4. Tokenise

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=MAX_LEN):
        self.texts = [str(t).strip() for t in texts]
        self.labels = labels
        self.encodings = tokenizer(
            self.texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt',
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_loader = DataLoader(ReviewDataset(train_texts, train_labels),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ReviewDataset(val_texts, val_labels), batch_size=BATCH_SIZE)
test_loader = DataLoader(ReviewDataset(test_df['reviews'].tolist()),
                         batch_size=BATCH_SIZE)

## 5. Train

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

# Question (f) wants this number: the parameter cost against 20,001 for the
# linear models on TF-IDF.
N_PARAMS = sum(p.numel() for p in model.parameters())
print(f'{N_PARAMS:,} parameters')

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

In [ ]:
from tqdm.auto import tqdm

BEST_PATH = 'best_distilbert_20split.pt'   # .pt is gitignored - never commit weights
best_val_f1 = 0.0
train_seconds = 0.0

for epoch in range(EPOCHS):
    model.train()
    started = time.perf_counter()
    total_train_loss = 0.0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False):
        optimizer.zero_grad()
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            labels=batch['labels'].to(DEVICE),
        )
        out.loss.backward()
        total_train_loss += out.loss.item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    train_seconds += time.perf_counter() - started

    model.eval()
    preds, truth = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]', leave=False):
            logits = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
            ).logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            truth.extend(batch['labels'].numpy())

    acc = accuracy_score(truth, preds)
    macro = f1_score(truth, preds, average='macro')
    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_train_loss/len(train_loader):.4f} '
          f'| Val Acc: {acc:.4f} | Val Macro F1: {macro:.4f}')

    if macro > best_val_f1:
        best_val_f1 = macro
        best_epoch = epoch + 1
        torch.save(model.state_dict(), BEST_PATH)
        best_preds, best_truth = list(preds), list(truth)

## 6. Full metrics for the best epoch

Accuracy alone is not reportable here - predicting 'positive' for everything
scores 0.853. Per-class precision, recall and F1 are what questions (c) to (f)
are marked on.

In [ ]:
report = classification_report(
    best_truth, best_preds, target_names=['negative', 'positive'],
    output_dict=True, zero_division=0,
)
print(classification_report(best_truth, best_preds,
                            target_names=['negative', 'positive'], zero_division=0))
cm = confusion_matrix(best_truth, best_preds, labels=[0, 1])
print('Confusion matrix (rows = true, cols = predicted):')
print(cm)

## 7. Write `results/distilbert_metrics.json`

This is the handoff. `src/compare_models.py` prefers this file over scraping
`03_distilbert.ipynb`, and because `test_size` is 0.2 it marks the row
`comparable: true` - so DistilBERT joins the main table and
`fig08_model_comparison.png` loses its hatching.

After running this, back in the repo:

```bash
python src/compare_models.py
python src/make_comparison_figures.py
```

In [ ]:
metrics = {
    'name': 'DistilBERT (4 epochs, max_length=256)',
    'short': 'DistilBERT',
    'owner': 'Ryan',
    'source': 'notebooks/04_distilbert_20split.ipynb',
    'test_size': TEST_SIZE,
    'accuracy': float(accuracy_score(best_truth, best_preds)),
    'macro_f1': float(f1_score(best_truth, best_preds, average='macro')),
    'negative_precision': report['negative']['precision'],
    'negative_recall': report['negative']['recall'],
    'negative_f1': report['negative']['f1-score'],
    'positive_precision': report['positive']['precision'],
    'positive_recall': report['positive']['recall'],
    'positive_f1': report['positive']['f1-score'],
    'confusion_matrix': {
        'tn': int(cm[0, 0]), 'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]), 'tp': int(cm[1, 1]),
    },
    'train_seconds': round(train_seconds, 1),
    'parameters': int(N_PARAMS),
    'config': {
        'model': MODEL_NAME,
        'max_length': MAX_LEN,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup': '10% of total steps',
        'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu',
        'peak_gpu_memory_gb': (round(torch.cuda.max_memory_allocated() / 1e9, 2)
                               if DEVICE.type == 'cuda' else None),
        'best_epoch': best_epoch,
    },
    'why': ('Contextual subword embeddings instead of a bag of words, so negation '
            'and word order survive, at far higher parameter and compute cost.'),
}

out_path = RESULTS_DIR / 'distilbert_metrics.json'
with open(out_path, 'w', encoding='utf-8', newline='\n') as f:
    json.dump(metrics, f, indent=2)
    f.write('\n')
print('wrote', out_path)
print(json.dumps(metrics, indent=2)[:600])

## 8. Test predictions

Written to `results/submission_distilbert_20split.csv` so it never silently
overwrites either `submission.csv` or Ryan's original
`submission_distilbert.csv`. Deciding which file is submitted is a group call.

In [ ]:
model.load_state_dict(torch.load(BEST_PATH))
model.eval()

test_predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test'):
        logits = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
        ).logits
        test_predictions.extend(torch.argmax(logits, dim=1).cpu().numpy())

submission = pd.DataFrame({
    'reviews': test_df['reviews'],
    'predictions': [int(p) for p in test_predictions],   # the course column name
})
sub_path = RESULTS_DIR / 'submission_distilbert_20split.csv'
submission.to_csv(sub_path, index=False, lineterminator='\n')
print('wrote', sub_path, '|', len(submission), 'rows')
print(submission['predictions'].value_counts().to_dict())

## 9. Download the results

Colab deletes its files when the session ends, so this step is essential. It
downloads one zip holding `distilbert_metrics.json` and the predictions. Also use
`File` -> `Download` -> `Download .ipynb` to keep the printed outputs as evidence.

In [ ]:
import shutil

archive = shutil.make_archive('distilbert_20split_results', 'zip', RESULTS_DIR)
print('packed', archive)
if IN_COLAB:
    files.download(archive)